# Geocoder

One notebook, two interchangeable backends.

| | Nominatim (OpenStreetMap) | Google Geocoding API |
|---|---|---|
| Cost | Free | 10,000 free calls/month, then $5.00 / 1,000 |
| Rate | 1 request/second (enforced) | ~50 req/s (throttled to 5 here) |
| Setup | `NOMINATIM_USER_AGENT` in `.env` | `GOOGLE_MAPS_API_KEY` **and** `GOOGLE_GEOCODING_CONFIRM=1` |

Every result is cached in `.cache/geocode.sqlite`, so re-running this notebook
costs **zero** provider calls. `MAX_REQUESTS_PER_RUN` caps live calls per run.


# Import geocoding tools

In [ ]:
import pandas as pd

from geocoding_tool import (
    build_query,
    geocode_dataframe,
    get_geocoder,
    load_env,
    project_root,
    to_geodataframe,
    write_attribution,
)

load_env()  # reads .env (copy .env.example first)

ROOT = project_root()
INPUT_DIR = ROOT / "data" / "input"
OUTPUT_DIR = ROOT / "data" / "output"

## Choose a backend

This is the only line you change to switch providers.

In [ ]:
PROVIDER = "nominatim"  # "nominatim" | "google"

INPUT_FILE = "dim_organizational_structure.csv"
OUTPUT_FILE = "geocoded.csv"

# Columns joined into the query string, most specific first.
# Keep these to *place* names. Adding an organisation column such as
# `district` ("Agder Røde Kors") makes Nominatim miss every row -- verified
# against the live service: 0/5 matched with it, 5/5 without.
QUERY_COLUMNS = ["organizational_town"]
COUNTRY_SUFFIX = "Norway"  # set to None to leave the query unqualified

geocoder = get_geocoder(PROVIDER)
print(f"provider: {geocoder.name}")
print(f"budget:   {geocoder.budget.limit} live calls this run")
print(f"cache:    {len(geocoder.cache)} results already stored")

# Read data

In [ ]:
df = pd.read_csv(INPUT_DIR / INPUT_FILE, dtype="string")
print(f"{len(df)} rows, {len(df.columns)} columns")
df.head()

## Build the query column

`build_query` joins the chosen columns per row, dropping blanks and literal
`"null"` values so a missing town never produces a broken query.

Inspect the result before geocoding: a query that carries an organisation name
rather than a place will simply miss, and each miss still costs a request.

In [ ]:
df["query"] = build_query(df, QUERY_COLUMNS, suffix=COUNTRY_SUFFIX)

n_unique = df["query"].nunique()
print(f"{len(df)} rows -> {n_unique} unique queries")
print(f"at most {n_unique} live calls (fewer if already cached)")
df[["local_branch", "query"]].head()

## Dry run first

Uncomment to work on a slice while you tune the query columns — especially
worth doing before pointing this at Google.

In [ ]:
# df = df.head(5)  # <- uncomment to test on 5 rows first

# Geocode

Nominatim runs at 1 request/second by policy, so a few hundred unique queries
take a few minutes. Failures land in `geo_error` rather than stopping the run;
hitting `MAX_REQUESTS_PER_RUN` raises `GeocodeBudgetExceeded` and stops it.

In [ ]:
result = geocode_dataframe(df, "query", geocoder)
result.head()

## Inspect the failures

In [ ]:
failed = result[result["geo_error"].notna()]
print(f"{len(failed)} of {len(result)} rows unresolved")

if not failed.empty:
    display(failed["geo_error"].value_counts())
    display(failed[["query", "geo_error"]].head(20))

## Geometry

WGS84 (EPSG:4326) points, ready for spatial joins or a map.

In [ ]:
gdf = to_geodataframe(result)
print(f"{len(gdf)} geocoded rows")
gdf.plot(figsize=(6, 8), markersize=8)

# Export

The attribution sidecar is written automatically. OpenStreetMap data is ODbL
licensed — that notice must accompany anything you publish from it.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
out_path = OUTPUT_DIR / OUTPUT_FILE

result.to_csv(out_path, index=False)
gdf.to_file(out_path.with_suffix(".gpkg"), driver="GPKG")
sidecar = write_attribution(out_path, geocoder)

print(f"wrote {out_path}")
print(f"wrote {out_path.with_suffix('.gpkg')}")
print(f"wrote {sidecar}")
print()
print(geocoder.attribution)